# Primeros pasos / First steps

Este notebook muestra cómo cargar el corpus y empezar a analizarlo: primero los metadatos (autores, órdenes, fechas) con pandas; después los textos (frecuencias, concordancias, vocabulario característico); por último, cómo pasar una selección de textos a [Voyant Tools](https://voyant-tools.org/).

*This notebook shows how to load the corpus and start analyzing it: the metadata with pandas, the texts themselves, and how to send a selection to Voyant Tools.*

Funciona en Google Colab o en tu computadora, con Python 3 y las bibliotecas `pandas`, `matplotlib` y `scikit-learn`.

## 1. Cargar el corpus / Load the corpus

Si el notebook está dentro de la carpeta del repositorio (descargada o clonada), usa esos archivos. Si no, por ejemplo en Colab, descarga el repositorio (unos 400 MB, un par de minutos).

In [ ]:
import os, subprocess

if not os.path.exists("metadatos/textos.csv"):
    if not os.path.exists("catholic-pastoral-texts"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vegafabianr/catholic-pastoral-texts.git"], check=True)
    os.chdir("catholic-pastoral-texts")
print("Corpus listo en la carpeta", os.path.basename(os.getcwd()))

In [ ]:
import pandas as pd

texts = pd.read_csv("metadatos/textos.csv")        # one row per text: the work, its author, its first edition...
editions = pd.read_csv("metadatos/ediciones.csv")  # one row per text: the 19th- or 20th-century edition used
authors = pd.read_csv("metadatos/autores.csv")     # one row per author
print(len(texts), "textos,", texts["work_id"].nunique(), "obras,", f"{texts['words'].sum():,}", "palabras")
texts.head(3)

## 2. Análisis bibliográfico con pandas / Bibliographic analysis

Cada fila de `textos.csv` es un texto (un archivo). Una obra en varios tomos tiene varias filas con el mismo `work_id`; para contar obras y no tomos se usa `drop_duplicates("work_id")`.

In [ ]:
works = texts.drop_duplicates("work_id")

# Obras por orden religiosa del autor / Works by religious order of the author
works["author_institution"].value_counts()

In [ ]:
# Obras por tema y género (solo los géneros con más de dos obras) / Works by topic and genre
table = pd.crosstab(works["topic"], works["genre"])
table.loc[:, table.sum() > 2]

In [ ]:
import matplotlib.pyplot as plt

# Cuándo se imprimieron por primera vez las obras, por década / When the works were first printed, by decade
decades = (works["first_edition"] // 10 * 10).value_counts().sort_index()
decades.plot(kind="bar", figsize=(12, 3), color="#3a7d5c", title="Obras por década de primera edición")
plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Filtros combinados: obras de jesuitas impresas por primera vez en el siglo XVII
# Combined filters: Jesuit works first printed in the seventeenth century
jesuits_1600s = works[(works["author_institution"] == "Jesuita") & works["first_edition"].between(1601, 1700)]
jesuits_1600s[["author", "title", "first_edition", "original_language", "translator"]].sort_values("first_edition")

In [ ]:
# Unir con la tabla de ediciones: de qué edición sale cada texto y qué tan bueno es su OCR
# Join with the editions: where each text comes from and how good its OCR is
full = texts.merge(editions, on="text_id")
full[["text_id", "edition_year", "edition_city", "unknown_words_pct"]].sort_values("unknown_words_pct").tail(5)

## 3. Análisis textual / Text analysis

Los textos son archivos `.txt` en UTF-8. Conservan unas pocas marcas (`<pb n="12"/>` salto de página, `<head>` título, `<small>` letra chica, `<foreign xml:lang="la">` latín); la función `read_text` las quita.

In [ ]:
import re
from collections import Counter

def read_text(path, keep_latin=False):
    with open(path, encoding="utf-8") as f:
        text = f.read()
    if not keep_latin:
        text = re.sub(r"<foreign[^>]*>.*?</foreign>", " ", text, flags=re.S)  # drop Latin passages
    return re.sub(r"<[^>]+>", " ", text)                                      # drop the other tags

def words(text):
    return re.findall(r"[^\W\d_]+", text.lower())

granada = read_text("textos-modernizados/luis-de-granada-guia-de-pecadores.txt")
print(granada[:600])

In [ ]:
# Palabras más frecuentes, sin palabras vacías / Most frequent words, without stopwords
STOP = set('''de la que el en y a los se no las por con su del lo como es para más al una un sus le mas o ni porque
esto este esta pues si ser sino así todo todos cuando también qué cual muy nos ha hay son era fue sin tan otra otros
otro donde ya mismo cosas cosa sobre entre dice todas estas aquí ella ellos ellas tiene está están quien aquel aquella
aquellos tanto estos tiene tienen había han hace hacer sea fuera puede vez cada aun después antes luego'''.split())
counts = Counter(w for w in words(granada) if w not in STOP and len(w) > 2)
counts.most_common(20)

In [ ]:
# Concordancia: cada aparición de una palabra con su contexto / Keyword in context
def kwic(text, word, width=60, limit=10):
    for m in list(re.finditer(rf"\b{word}\b", text, flags=re.I))[:limit]:
        left = text[max(0, m.start() - width):m.start()].replace("\n", " ")
        right = text[m.end():m.end() + width].replace("\n", " ")
        print(f"{left:>{width}} [{m.group()}] {right}")

kwic(granada, "conciencia")

In [ ]:
# Vocabulario característico de dos grupos de textos, con TF-IDF / Distinctive vocabulary of two groups
from sklearn.feature_extraction.text import TfidfVectorizer

group_a = texts[texts["author_institution"] == "Jesuita"]
group_b = texts[texts["author_institution"] == "Carmelita"]
docs = [read_text(path) for path in pd.concat([group_a, group_b])["modern_file"]]
vectorizer = TfidfVectorizer(min_df=3, max_df=0.8, token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
matrix = vectorizer.fit_transform(docs)
mean_a = matrix[:len(group_a)].mean(axis=0).A1
mean_b = matrix[len(group_a):].mean(axis=0).A1
difference = pd.Series(mean_a - mean_b, index=vectorizer.get_feature_names_out()).sort_values()
print("Más en carmelitas:", ", ".join(difference.index[:20]))
print("Más en jesuitas:  ", ", ".join(difference.index[-20:][::-1]))

## 4. Llevar textos a Voyant Tools

[Voyant Tools](https://voyant-tools.org/) es una herramienta web de análisis textual que no requiere programar. Hay dos maneras de cargarle textos:

1. **Por enlace**, sin descargar nada: la celda siguiente imprime los enlaces de una selección. Se copian, se pegan en el recuadro de la página de inicio de Voyant (uno por línea) y se pulsa *Reveal*.
2. **Por archivo**: la otra celda guarda la selección, sin marcas, en `seleccion-voyant.zip`, que se sube con el botón *Upload* de Voyant.

El corpus completo ya está cargado en Voyant, sin tildes: [abrirlo](https://voyant-tools.org/?corpus=9b3cdbaeec7055ab26e73a1352736732). Lo que sigue sirve para trabajar con una selección.

In [ ]:
BASE = "https://raw.githubusercontent.com/vegafabianr/catholic-pastoral-texts/main/"
selection = texts[texts["author"] == "Teresa de Ávila"]
for path in selection["modern_file"]:
    print(BASE + path)

In [ ]:
import zipfile

with zipfile.ZipFile("seleccion-voyant.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in selection["modern_file"]:
        z.writestr(os.path.basename(path), read_text(path, keep_latin=True))
print("Guardado: seleccion-voyant.zip, en la carpeta del repositorio")